# Guardrailed Multi-Agent System: run and evidence notebook

This notebook runs the full system on a Google Colab and records the evidence for the submission.

* **Part 1** prepares the Colab computer (download the project, install libraries, add the key, run the tests).
* **Part 2** is session 1: save organisation facts, run a normal job, show the attempt cap and the kill-switch.
* **Part 3** is session 2, after a restart: prove the memory survived and is used.
* **Part 4** shows the same engine reused for a different kind of job.

Run the cells from top to bottom. Every command runs as a separate program, so each one is also a fresh session.

## Part 1: prepare the Colab computer

### 1.1 Download (clone) the project from GitHub
Change `GITHUB_USER` if your GitHub username is different.

In [ ]:
GITHUB_USER = "mapanya"
REPO_NAME = "guardrailed-multi-agent-system"
REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"

import os
os.chdir("/content")
if os.path.isdir(REPO_NAME):
    !cd {REPO_NAME} && git pull
else:
    !git clone {REPO_URL}
os.chdir(f"/content/{REPO_NAME}")
!pwd && ls

### 1.2 Install the libraries
This takes 2 to 4 minutes. Warnings about other Colab packages can be ignored.

In [ ]:
!python --version
!pip install -q -r requirements.txt
!pip install -q -e .
print("Install finished")

### 1.3 Add your Groq key
The cell first looks for a Colab secret called `GROQ_API_KEY`. If it cannot find one, it asks you to paste the key into a hidden box. The key is kept only in this session and is never written to a file.

In [ ]:
import os
from getpass import getpass

def load_key():
    try:
        from google.colab import userdata
        key = userdata.get("GROQ_API_KEY")
        if key:
            return key
    except Exception:
        pass
    return getpass("Paste your Groq key and press Enter: ")

os.environ["GROQ_API_KEY"] = load_key().strip()
print("Key loaded:", "yes" if os.environ["GROQ_API_KEY"] else "no")

### 1.4 Run the automated tests
These check the guardrails, the output forms and the workflow without using the key. All should pass.

In [ ]:
!python -m pytest -q

## Part 2: session 1

### 2.1 Save organisation facts to memory
These facts are stored in the SQLite history file and the local search index inside the `data` folder.

In [ ]:
USER_ID = "demo_user"
!python -m guardrailed_mas.main remember --user-id {USER_ID} --fact "Our email platform is Microsoft 365 with Defender for Office 365."
!python -m guardrailed_mas.main remember --user-id {USER_ID} --fact "All security incidents are escalated to the SOC Lead and the IT Manager."

### 2.2 Normal run: phishing incident
Watch the step lines: `planner`, `budget_gate` with `execution_count`, `executor`, `reviewer` and finally `finalize` (approved) or `fallback` (halted).

In [ ]:
!python -m guardrailed_mas.main run --scenario phishing --user-id {USER_ID}

### 2.3 Guardrail demonstration: the Reviewer rejects everything
The Executor is allowed 3 attempts. When `execution_count` reaches 4 (more than 3), the graph stops at once and returns the fixed fallback answer.

In [ ]:
!python -m guardrailed_mas.main run --scenario ransomware --user-id {USER_ID} --demo-guardrail

### 2.4 Kill-switch demonstration
With the emergency stop switched on, no worker is called at all.

In [ ]:
!MAS_KILL_SWITCH=1 python -m guardrailed_mas.main run --scenario lost_laptop --user-id {USER_ID}

## Part 3: session 2 (after a restart)

**Now restart the session:** in Colab choose *Runtime > Restart session*. In VS Code click **Restart** on the notebook toolbar.

A restart wipes everything held in the notebook's working memory, including the key and all variables. The `data` folder on the Colab computer is kept. That is where Mem0 stored its SQLite file. Then run the cells below.

In [ ]:
import os
from getpass import getpass
os.chdir("/content/guardrailed-multi-agent-system")

def load_key():
    try:
        from google.colab import userdata
        key = userdata.get("GROQ_API_KEY")
        if key:
            return key
    except Exception:
        pass
    return getpass("Paste your Groq key and press Enter: ")

os.environ["GROQ_API_KEY"] = load_key().strip()
USER_ID = "demo_user"
print("Session 2 ready")

### 3.1 What does memory hold now?
The facts from session 1 and short run summaries are still there.

In [ ]:
!python -m guardrailed_mas.main recall --user-id {USER_ID}

### 3.2 Look inside the SQLite file directly

In [ ]:
import sqlite3
db = "data/mem0_history.db"
with sqlite3.connect(db) as con:
    rows = con.execute("SELECT event, substr(new_memory, 1, 90), created_at FROM history ORDER BY created_at").fetchall()
print(f"{len(rows)} row(s) in {db}")
for row in rows:
    print(row)

### 3.3 New job that uses the remembered facts
In the step lines, `recall_memory` shows `facts_found` above 0 and the checklist should mention Microsoft 365 and the SOC lead without being told again.

In [ ]:
!python -m guardrailed_mas.main run --scenario account_takeover --user-id {USER_ID}

## Part 4: reuse for a different job
The engine is unchanged. Only the worker profile in `config/agents.yaml` differs.

In [ ]:
!python -m guardrailed_mas.main run --scenario new_customer --user-id {USER_ID}

You can also describe any job in your own words:

In [ ]:
!python -m guardrailed_mas.main run --task "A contractor reports that a USB drive with project files is missing." --user-id {USER_ID}

## Part 5: collect the evidence
This prints every step log. Save this notebook with its outputs (File > Save). The saved notebook and your screenshots are the run evidence for GitHub.

In [ ]:
import glob
for path in sorted(glob.glob("logs/run_*.jsonl")):
    print("=" * 20, path)
    print(open(path).read())